# Fed Chair speeches/testimony -- update

Re-runs discovery for every speaker (cheap -- scrape_speaker skips any PDF
already on disk) and appends any item_id not already in the saved manifest.
Run this periodically to pick up new speeches from active officials
(e.g. Powell, Jefferson).

In [1]:
import os, sys
import pandas as pd
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
import fed_speeches_helpers as fsh
from pathlib import Path

BASE = os.path.dirname(os.getcwd())
DOWNLOADS_DIR = Path(r"C:\Users\User\Desktop\BS ANALYSIS\FEDSCRAPING\speeches_chair\downloads")
MANIFEST_PATH = os.path.join(BASE, "Data Input", "Raw Input", "fed_speeches_manifest_raw.csv")

## Where are we

In [2]:
existing = pd.read_csv(MANIFEST_PATH)
print(f"{len(existing):,} items on file")

2,087 items on file


## Re-discover + download anything new

In [3]:
manifests = []
for label, title_id in fsh.CHAIR_IDS.items():
    m = fsh.scrape_speaker(label, title_id, DOWNLOADS_DIR)
    manifests.append(m)
    if m.attrs["dl_ok"] > 0:
        print(f"{label}: {m.attrs['dl_ok']} new file(s) downloaded")

fresh = pd.concat(manifests, ignore_index=True)
new_rows = fresh[~fresh["item_id"].isin(existing["item_id"])]
print(f"{len(new_rows)} new item(s) found")

0 new item(s) found


## Append and save

In [4]:
if len(new_rows):
    updated = pd.concat([existing, new_rows], ignore_index=True).drop_duplicates(subset=["item_id", "path"])
    updated.to_csv(MANIFEST_PATH, index=False)
    print(f"Appended {len(new_rows)} row(s), {len(updated):,} total")
else:
    print("Nothing new.")

Nothing new.
